In [3]:
import pandas as pd
import numpy as np
import re
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

# ==========================================
# 1. 读取原始数据与战术特征工程 (Data Cleaning)
# ==========================================
print("⏳ 正在读取原始数据 data.csv 并进行特征提取...")

# 请确保原始的 data.csv 文件在当前运行目录下
df = pd.read_csv('/data.csv')

rows = []

for idx, row in df.iterrows():
    clean_score = re.sub(r'\(.*?\)', '', str(row['score'])).replace('–', '-').strip()
    parts = clean_score.split('-')
    h_score, a_score = int(parts[0].strip()), int(parts[1].strip())

    h_pass_acc = round((row['home_completed_passes'] / row['home_attempted_pases']) * 100, 2) if row['home_attempted_pases'] > 0 else 0
    a_pass_acc = round((row['away_completed_passes'] / row['away_attempted_pases']) * 100, 2) if row['away_attempted_pases'] > 0 else 0

    h_ppda = round(row['away_completed_passes'] / max(row['home_tackles'] + row['home_interceptions'], 1), 2)
    a_ppda = round(row['home_completed_passes'] / max(row['away_tackles'] + row['away_interceptions'], 1), 2)

    total_aerials = max(row['home_aerials_won'] + row['away_aerials_won'], 1)
    h_aerial = round((row['home_aerials_won'] / total_aerials) * 100, 2)
    a_aerial = round((row['away_aerials_won'] / total_aerials) * 100, 2)

    # 主队视角记录
    h_outcome = 'Win' if h_score > a_score else ('Loss' if h_score < a_score else 'Draw')
    rows.append({
        'match_id': row['match'], 'team': row['home_team'], 'opponent': row['away_team'],
        'xg': float(row['home_xg']), 'possession': float(row['home_possession']),
        'shots_on_target': float(row['home_sot']), 'shots_total': float(row['home_total_shots']),
        'passes_completed': float(row['home_completed_passes']), 'pass_accuracy': h_pass_acc,
        'ppda': h_ppda, 'tackles_successful': float(row['home_tackles']),
        'interceptions': float(row['home_interceptions']), 'clearances': float(row['home_clearances']),
        'fouls_committed': float(row['home_fouls']), 'yellow_cards': 0,
        'corners': float(row['home_corners']), 'crosses_completed': float(row['home_crosses']),
        'aerial_duels_won_pct': h_aerial, 'errors_leading_to_shot': 0, 'outcome': h_outcome
    })

    # 客队视角记录
    a_outcome = 'Win' if a_score > h_score else ('Loss' if a_score < h_score else 'Draw')
    rows.append({
        'match_id': row['match'], 'team': row['away_team'], 'opponent': row['home_team'],
        'xg': float(row['away_xg']), 'possession': float(row['away_possession']),
        'shots_on_target': float(row['away_sot']), 'shots_total': float(row['away_total_shots']),
        'passes_completed': float(row['away_completed_passes']), 'pass_accuracy': a_pass_acc,
        'ppda': a_ppda, 'tackles_successful': float(row['away_tackles']),
        'interceptions': float(row['away_interceptions']), 'clearances': float(row['away_clearances']),
        'fouls_committed': float(row['away_fouls']), 'yellow_cards': 0,
        'corners': float(row['away_corners']), 'crosses_completed': float(row['away_crosses']),
        'aerial_duels_won_pct': a_aerial, 'errors_leading_to_shot': 0, 'outcome': a_outcome
    })

# 保存为标准格式供 Streamlit 前端读取
clean_df = pd.DataFrame(rows)
clean_df.to_csv('clean_world_cup_2022.csv', index=False)
print("✅ 第一步：数据清洗完毕！成功导出 clean_world_cup_2022.csv (共 128 条记录)。")

# ==========================================
# 2. 特征瘦身与模型抗压测试 (7 Golden Features)
# ==========================================
print("\n⏳ 正在进行模型瘦身与 5-Fold 交叉验证测试...")

golden_features = [
    'xg', 'possession', 'shots_on_target',
    'ppda', 'tackles_successful', 'interceptions', 'aerial_duels_won_pct'
]

X = clean_df[golden_features]
y = clean_df['outcome']

rf_model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(rf_model, X, y, cv=cv, scoring='accuracy')

print(f"🎯 第二步：排除噪音后，5-Fold Cross-Validation 真实平均准确率: {scores.mean() * 100:.2f}%")

# ==========================================
# 3. 打包新一代战术大脑 (Export v2 pkl)
# ==========================================
rf_model.fit(X, y)
joblib.dump(rf_model, 'world_cup_rf_model_v2.pkl')
print("\n✅ 第三步：新一代 AI 战术大脑已顺利导出为 world_cup_rf_model_v2.pkl！")

⏳ 正在读取原始数据 data.csv 并进行特征提取...
✅ 第一步：数据清洗完毕！成功导出 clean_world_cup_2022.csv (共 128 条记录)。

⏳ 正在进行模型瘦身与 5-Fold 交叉验证测试...
🎯 第二步：排除噪音后，5-Fold Cross-Validation 真实平均准确率: 37.32%

✅ 第三步：新一代 AI 战术大脑已顺利导出为 world_cup_rf_model_v2.pkl！
